In [1]:
# Complete setup - run all prerequisite code at once
import os
os.chdir('/net/scratch2/smallyan/pronoun_claude_2025-12-26_01-36-15')

# Load bashrc environment for cached models
import subprocess
result = subprocess.run(['bash', '-c', 'source ~/.bashrc && env'], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if '=' in line:
        key, _, value = line.partition('=')
        os.environ[key] = value

import torch
import numpy as np
import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')
import random
random.seed(42)
import pandas as pd
from functools import partial

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

from transformer_lens import HookedTransformer
model = HookedTransformer.from_pretrained("gpt2-small", device=device)
print(f"Model loaded: {model.cfg.model_name}")

# Define occupation lists
FEMALE_STEREOTYPED = [
    "nurse", "teacher", "secretary", "receptionist", "librarian",
    "housekeeper", "nanny", "hairdresser", "dietitian", "dental hygienist",
    "paralegal", "social worker", "cashier", "maid", "babysitter"
]
MALE_STEREOTYPED = [
    "engineer", "doctor", "programmer", "mechanic", "carpenter",
    "electrician", "plumber", "pilot", "surgeon", "CEO",
    "lawyer", "scientist", "firefighter", "police officer", "construction worker"
]
GENDER_NEUTRAL = [
    "student", "writer", "artist", "manager", "chef",
    "accountant", "consultant", "analyst", "designer", "journalist"
]
MALE_PRONOUNS = ["He", "he", "Him", "him", "His", "his"]
FEMALE_PRONOUNS = ["She", "she", "Her", "her"]

# Templates
TEMPLATES = [
    "The {occupation} finished the task. {pronoun} then left.",
    "The {occupation} arrived at work. {pronoun} started immediately.",
    "A {occupation} was hired yesterday. {pronoun} seemed competent.",
    "The {occupation} checked the schedule. {pronoun} nodded.",
    "The {occupation} completed the report. {pronoun} submitted it.",
    "The {occupation} entered the room. {pronoun} sat down.",
    "A {occupation} was called urgently. {pronoun} hurried over.",
    "The {occupation} reviewed the documents. {pronoun} found errors.",
]

print("Setup complete!")

Using device: cuda


`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer
Model loaded: gpt2
Setup complete!


In [2]:
# Create dataset
dataset = []

def create_example(occupation, pronoun, is_stereotypical, category):
    """Create a pronoun resolution example."""
    template = random.choice(TEMPLATES)
    text = template.format(occupation=occupation, pronoun=pronoun)
    tokens = model.to_tokens(text, prepend_bos=True)[0]
    str_tokens = model.to_str_tokens(text, prepend_bos=True)
    
    pronoun_pos = None
    for i, tok in enumerate(str_tokens):
        if tok.strip().lower() in ['he', 'she', 'him', 'her', 'his']:
            pronoun_pos = i
            break
    
    occupation_pos = None
    for i, tok in enumerate(str_tokens):
        if occupation.lower() in tok.lower():
            occupation_pos = i
            break
    
    return {
        'text': text, 'occupation': occupation, 'pronoun': pronoun,
        'is_stereotypical': is_stereotypical, 'category': category,
        'pronoun_gender': 'male' if pronoun.lower() in ['he', 'him', 'his'] else 'female',
        'tokens': str_tokens, 'pronoun_pos': pronoun_pos, 'occupation_pos': occupation_pos
    }

# Create dataset
for occ in FEMALE_STEREOTYPED:
    dataset.append(create_example(occ, 'She', True, 'female_stereo'))
for occ in MALE_STEREOTYPED:
    dataset.append(create_example(occ, 'He', True, 'male_stereo'))
for occ in FEMALE_STEREOTYPED:
    dataset.append(create_example(occ, 'He', False, 'counter_male'))
for occ in MALE_STEREOTYPED:
    dataset.append(create_example(occ, 'She', False, 'counter_female'))

# Neutral examples
for occ in GENDER_NEUTRAL:
    for pronoun in ['He', 'She']:
        template = random.choice(TEMPLATES)
        text = template.format(occupation=occ, pronoun=pronoun)
        tokens = model.to_str_tokens(text, prepend_bos=True)
        pronoun_pos = next((i for i, t in enumerate(tokens) if t.strip().lower() in ['he', 'she']), None)
        occupation_pos = next((i for i, t in enumerate(tokens) if occ.lower() in t.lower()), None)
        if pronoun_pos:
            dataset.append({'text': text, 'occupation': occ, 'pronoun': pronoun, 'is_stereotypical': None,
                'category': 'neutral', 'pronoun_gender': 'male' if pronoun == 'He' else 'female',
                'tokens': tokens, 'pronoun_pos': pronoun_pos, 'occupation_pos': occupation_pos})

# Minimal pairs
MINIMAL_PAIR_TEMPLATE = "The {occupation} walked in. {pronoun} looked around."
for occ in FEMALE_STEREOTYPED[:5] + MALE_STEREOTYPED[:5]:
    for pronoun in ['He', 'She']:
        text = MINIMAL_PAIR_TEMPLATE.format(occupation=occ, pronoun=pronoun)
        tokens = model.to_str_tokens(text, prepend_bos=True)
        pronoun_pos = next((i for i, t in enumerate(tokens) if t.strip().lower() in ['he', 'she']), None)
        occupation_pos = next((i for i, t in enumerate(tokens) if occ.lower() in t.lower()), None)
        is_stereo = (pronoun == 'She' and occ in FEMALE_STEREOTYPED) or (pronoun == 'He' and occ in MALE_STEREOTYPED)
        if pronoun_pos:
            dataset.append({'text': text, 'occupation': occ, 'pronoun': pronoun, 'is_stereotypical': is_stereo,
                'category': 'minimal_pair', 'pronoun_gender': 'male' if pronoun == 'He' else 'female',
                'tokens': tokens, 'pronoun_pos': pronoun_pos, 'occupation_pos': occupation_pos})

print(f"Dataset size: {len(dataset)}")

Dataset size: 100


In [3]:
# Evaluate all examples
def get_pronoun_logprobs(model, text, pronoun_pos):
    tokens = model.to_tokens(text, prepend_bos=True)
    with torch.no_grad():
        logits = model(tokens)
    pred_pos = pronoun_pos - 1
    logits_at_pos = logits[0, pred_pos, :]
    he_token = model.to_single_token(' He')
    she_token = model.to_single_token(' She')
    he_token_no_space = model.to_single_token('He')
    she_token_no_space = model.to_single_token('She')
    log_probs = torch.log_softmax(logits_at_pos, dim=-1)
    he_prob = max(log_probs[he_token].item(), log_probs[he_token_no_space].item())
    she_prob = max(log_probs[she_token].item(), log_probs[she_token_no_space].item())
    return {'he': he_prob, 'she': she_prob}

results = []
for example in tqdm(dataset, desc="Evaluating"):
    if example['pronoun_pos'] is None:
        continue
    try:
        probs = get_pronoun_logprobs(model, example['text'], example['pronoun_pos'])
        actual_gender = example['pronoun_gender']
        predicted_gender = 'female' if probs['she'] > probs['he'] else 'male'
        correct = (predicted_gender == actual_gender)
        gender_diff = probs['she'] - probs['he']
        results.append({**example, 'he_logprob': probs['he'], 'she_logprob': probs['she'],
            'predicted_gender': predicted_gender, 'correct': correct, 'gender_diff': gender_diff})
    except Exception as e:
        print(f"Error: {e}")

df = pd.DataFrame(results)
print(f"Evaluated {len(results)} examples")

Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

Evaluated 100 examples


In [4]:
# Create occupation pairs for patching
occupation_pairs = {}
for occ in FEMALE_STEREOTYPED[:5] + MALE_STEREOTYPED[:5]:
    pairs = df[(df['category'] == 'minimal_pair') & (df['occupation'] == occ)]
    if len(pairs) == 2:
        he_example = pairs[pairs['pronoun_gender'] == 'male'].iloc[0].to_dict()
        she_example = pairs[pairs['pronoun_gender'] == 'female'].iloc[0].to_dict()
        occupation_pairs[occ] = {'he': he_example, 'she': she_example}

he_token = model.to_single_token(' He')
she_token = model.to_single_token(' She')

print(f"Created {len(occupation_pairs)} occupation pairs")

# Cell 18: Activation patching across all heads and MLPs
def run_patching_experiment(model, text, pronoun_pos, n_layers=12, n_heads=12):
    """Run activation patching on all heads for a single example."""
    tokens = model.to_tokens(text, prepend_bos=True)
    pred_pos = pronoun_pos - 1
    
    with torch.no_grad():
        clean_logits, clean_cache = model.run_with_cache(tokens)
    
    clean_metric = clean_logits[0, pred_pos, she_token] - clean_logits[0, pred_pos, he_token]
    
    head_effects = torch.zeros(n_layers, n_heads)
    
    for layer in range(n_layers):
        for head in range(n_heads):
            def patch_hook(activation, hook, layer=layer, head=head):
                activation[:, :, head, :] = 0
                return activation
            
            with torch.no_grad():
                patched_logits = model.run_with_hooks(
                    tokens,
                    fwd_hooks=[(f'blocks.{layer}.attn.hook_z', patch_hook)]
                )
            
            patched_metric = patched_logits[0, pred_pos, she_token] - patched_logits[0, pred_pos, he_token]
            head_effects[layer, head] = (clean_metric - patched_metric).item()
    
    mlp_effects = torch.zeros(n_layers)
    for layer in range(n_layers):
        def patch_mlp_hook(activation, hook, layer=layer):
            activation[:, :, :] = 0
            return activation
        
        with torch.no_grad():
            patched_logits = model.run_with_hooks(
                tokens,
                fwd_hooks=[(f'blocks.{layer}.mlp.hook_post', patch_mlp_hook)]
            )
        
        patched_metric = patched_logits[0, pred_pos, she_token] - patched_logits[0, pred_pos, he_token]
        mlp_effects[layer] = (clean_metric - patched_metric).item()
    
    return head_effects, mlp_effects, clean_metric.item()

print("Patching function defined")

Created 10 occupation pairs
Patching function defined


In [5]:
# Cell 19: Run patching across multiple examples
all_head_effects = []
all_mlp_effects = []

test_examples = df[(df['category'].isin(['female_stereo', 'male_stereo']))].head(20)
print(f"Running patching on {len(test_examples)} examples...")

for idx, row in tqdm(test_examples.iterrows(), total=len(test_examples)):
    try:
        head_effects, mlp_effects, _ = run_patching_experiment(
            model, row['text'], row['pronoun_pos']
        )
        if row['pronoun_gender'] == 'male':
            head_effects = -head_effects
            mlp_effects = -mlp_effects
        
        all_head_effects.append(head_effects)
        all_mlp_effects.append(mlp_effects)
    except Exception as e:
        print(f"Error: {e}")

mean_head_effects = torch.stack(all_head_effects).mean(dim=0)
mean_mlp_effects = torch.stack(all_mlp_effects).mean(dim=0)

print("Done!")

Running patching on 20 examples...


  0%|          | 0/20 [00:00<?, ?it/s]